In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits

SOLEXS_PATH = "/kaggle/input/datasets/garvsaluja/bah-2026/2026/2026/AL1_SLX_L1_20260607_v1.0/AL1_SLX_L1_20260607_v1.0/SDD2/AL1_SOLEXS_20260607_SDD2_L1.lc/AL1_SOLEXS_20260607_SDD2_L1.lc"
HELIOS_A = "/kaggle/input/datasets/garvsaluja/bah-2026/helios/helios/HLS_20260607_000008_43177sec_lev1_V111/2026/06/07/HLS_20260607_000008_43177sec_lev1_V111/cdte/hel1os_cdte_spectra_cdte1.fits"
HELIOS_B = "/kaggle/input/datasets/garvsaluja/bah-2026/helios/helios/HLS_20260607_120000_43193sec_lev1_V111/2026/06/07/HLS_20260607_120000_43193sec_lev1_V111/cdte/hel1os_cdte_spectra_cdte1.fits"


# LOAD helios

def load_helios_session(path):
    hdul = fits.open(path)
    obs_start = pd.Timestamp(
        hdul[0].header["ISOSTART"]
    ).tz_localize("UTC")
    data = hdul[1].data
    rows = []
    for row in data:
        counts = np.array(row["COUNTS"])
        rows.append({
            "TIME":
                obs_start +
                pd.to_timedelta(
                    row["TSTART"],
                    unit="s"
                ),

            "total_flux":
                counts.sum(),

            "low_flux":
                counts[:170].sum(),

            "mid_flux":
                counts[170:340].sum(),

            "high_flux":
                counts[340:].sum()

        })

    return pd.DataFrame(rows)
hdul = fits.open(SOLEXS_PATH)
solexs_data = hdul[1].data
solexs_df = pd.DataFrame({
    "TIME":
        solexs_data["TIME"],

    "COUNTS":
        np.asarray(
            solexs_data["COUNTS"],
            dtype=np.float64
        )
})

solexs_df["TIME"] = pd.to_datetime(
    solexs_df["TIME"],
    unit="s",
    utc=True
)

solexs_df = (
    solexs_df
    .set_index("TIME")
    .sort_index()
)

solexs_20s = (
    solexs_df["COUNTS"]
    .resample("20s")
    .mean()
    .to_frame(
        name="solexs_counts"
    )
)
helios_A = load_helios_session(
    HELIOS_A
)

helios_B = load_helios_session(
    HELIOS_B
)

helios_full = pd.concat(
    [helios_A, helios_B],
    ignore_index=True
)

helios_full = (
    helios_full
    .sort_values("TIME")
    .reset_index(drop=True)
)

combined = pd.merge_asof(
    solexs_20s.reset_index(),
    helios_full.sort_values(
        "TIME"
    ),

    on="TIME",
    direction="nearest",
    tolerance=pd.Timedelta(
        "10s"
    )
)

combined = combined.dropna()

feature_cols = [
    "solexs_counts",
    "total_flux",
    "low_flux",
    "mid_flux",
    "high_flux"

]

WINDOW = 90      # 30 minutes
STRIDE = 1

X = []

for start in range(

    0,

    len(combined) - WINDOW,

    STRIDE

):

    end = start + WINDOW

    X.append(

        combined[
            feature_cols
        ].iloc[
            start:end
        ].values

    )

X = np.array(X)


# RESULTS

print("Combined shape:")
print(combined.shape)

print()

print("X shape:")
print(X.shape)

print()

print("Feature order:")
print(feature_cols)

In [ ]:
import os
import re
import numpy as np
import pandas as pd

from astropy.io import fits

In [ ]:
solexs_index = {}

for root, dirs, files in os.walk("/kaggle/input"):

    for f in files:

        if (
            f.startswith("AL1_SOLEXS_")
            and f.endswith(".lc")
        ):

            match = re.search(
                r"(\d{8})",
                f
            )

            if match:

                date = match.group(1)

                solexs_index[date] = os.path.join(
                    root,
                    f
                )

helios_index = {}

for root, dirs, files in os.walk("/kaggle/input"):

    for f in files:

        if f == "hel1os_cdte_spectra_cdte1.fits":

            match = re.search(
                r"HLS_(\d{8})_(\d{6})",
                root
            )

            if match:

                date = match.group(1)

                start_time = match.group(2)

                helios_index.setdefault(
                    date,
                    []
                )

                helios_index[date].append(

                    (
                        start_time,
                        os.path.join(root, f)
                    )

                )

for date in helios_index:

    helios_index[date].sort()

    helios_index[date] = [

        x[1]

        for x in helios_index[date]

    ]

common_dates = sorted(

    set(solexs_index.keys())

    &

    set(helios_index.keys())

)

print(
    "Common Dates:",
    len(common_dates)
)

In [ ]:
def build_day_dataset(date, window=90):
    solexs_path = solexs_index[date]
    hdul = fits.open(solexs_path)
    data = hdul[1].data
    if "TIME" in data.dtype.names:

        solexs_df = pd.DataFrame({

            "TIME": pd.to_datetime(
                data["TIME"],
                unit="s",
                utc=True
            ),

            "COUNTS": np.asarray(
                data["COUNTS"],
                dtype=np.float64
            )

        })

    else:

        solexs_df = pd.DataFrame({

            "TIME": pd.to_datetime(
                data["TSTART"],
                unit="s",
                utc=True
            ),

            "COUNTS": [
                np.sum(spec)
                for spec in data["COUNTS"]
            ]

        })

    solexs_df = (
        solexs_df
        .set_index("TIME")
        .sort_index()
    )

    solexs_20s = (
        solexs_df["COUNTS"]
        .resample("20s")
        .mean()
        .to_frame(name="solexs_counts")
    )

    helios_parts = []

    for path in helios_index[date]:

        hdul = fits.open(path)

        obs_start = pd.Timestamp(
            hdul[0].header["ISOSTART"]
        ).tz_localize("UTC")

        data = hdul[1].data

        rows = []

        for row in data:

            counts = np.asarray(
                row["COUNTS"],
                dtype=np.float64
            )

            rows.append({

                "TIME":
                    obs_start +
                    pd.to_timedelta(
                        row["TSTART"],
                        unit="s"
                    ),

                "total_flux":
                    counts.sum(),

                "low_flux":
                    counts[:170].sum(),

                "mid_flux":
                    counts[170:340].sum(),

                "high_flux":
                    counts[340:].sum()

            })

        helios_parts.append(
            pd.DataFrame(rows)
        )

    helios_full = pd.concat(
        helios_parts,
        ignore_index=True
    )

    helios_full = (
        helios_full
        .sort_values("TIME")
        .reset_index(drop=True)
    )



    combined = pd.merge_asof(

        solexs_20s.reset_index(),

        helios_full,

        on="TIME",

        direction="nearest",

        tolerance=pd.Timedelta("10s")

    )

    combined = combined.dropna()
    feature_cols = [

        "solexs_counts",

        "total_flux",

        "low_flux",

        "mid_flux",

        "high_flux"

    ]

    X = []

    for start in range(
        len(combined) - window
    ):

        end = start + window

        X.append(

            combined[
                feature_cols
            ].iloc[
                start:end
            ].values

        )

    if len(X) == 0:

        X = np.empty(
            (
                0,
                window,
                len(feature_cols)
            )
        )

    else:

        X = np.array(X)

    return X, combined

In [ ]:
all_chunks = []

successful_dates = []
failed_dates = []

for date in common_dates:

    try:

        X_day, _ = build_day_dataset(date)

        if X_day.shape[0] > 0:

            all_chunks.append(X_day)

            successful_dates.append(date)

    except Exception as e:

        failed_dates.append(
            (date, str(e))
        )

print(
    "Successful:",
    len(successful_dates)
)

print(
    "Failed:",
    len(failed_dates)
)

X_all = np.concatenate(
    all_chunks,
    axis=0
)

X_all = X_all.astype(
    np.float32
)

print(
    "Final Shape:",
    X_all.shape
)

np.save(
    "X_all.npy",
    X_all
)

print("Dataset Saved")